In [ ]:
!pip install openpyxl
!pip install pydrive --upgrade

from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
import pandas as pd
import openpyxl
import requests
import numpy as np
from io import BytesIO
import datetime as dt
creds, _ = default()
import regex as re


gc = gspread.authorize(creds)

# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
headers = ["Work Type", "Practice", "Claim ID", "Worked By", "Worked Date", "Work Status"]

listOfFrames = []

conn = gspread.authorize(creds)
sheets = [
          '1msfbrNKRF8iDvLxwS9TgvJKVibZXt8XLqcFW_qnOylc',       #Claims Billing Tracker - Nov 2025
          '1RPOinQZG4QQjsxhr2Zsi-kwmQxR7dAMlYpvqDYJGV6E',       #Claims Billing Tracker - Dec 2025
          '172ZNGOfYtDtjEqMLPKaaPgSqFufO4vfuzItLelWjIl4'   ]    #Claims Billing Tracker - Jan 2026

for sheet in sheets:  # Added
    worksheet_list = conn.open_by_key(sheet).worksheet("After Transmit/Manual Exp/Tickets")
    rows = worksheet_list.get_all_values()
    data = zip(*(e for e in zip(*rows) if e[0].strip() in headers))
    df = pd.DataFrame(data, columns=headers)
    df.rename(columns = df.iloc[0].apply(lambda x: x.strip()), inplace = True)
    df.drop(df.index[0], inplace = True)
    listOfFrames.append(df)
    print(sheet)

1msfbrNKRF8iDvLxwS9TgvJKVibZXt8XLqcFW_qnOylc
1RPOinQZG4QQjsxhr2Zsi-kwmQxR7dAMlYpvqDYJGV6E
172ZNGOfYtDtjEqMLPKaaPgSqFufO4vfuzItLelWjIl4


In [ ]:
corrected_listOfFrames = []
for df in listOfFrames:
    # Ensure the DataFrame has the correct number of columns before reassigning headers
    # The kernel state shows DataFrames having 6 columns, which matches len(headers)
    if len(df.columns) == len(headers):
        df.columns = headers
    else:
        # Handle cases where column count might unexpectedly differ
        # For now, we assume the count is consistent and the issue is just non-unique naming.
        # If this branch is hit, a more complex transformation would be needed.
        print(f"Warning: DataFrame column count mismatch. Expected {len(headers)}, got {len(df.columns)}.")
        # Attempt to make columns unique if mismatched, then assign.
        # This is a fallback and might require more specific logic depending on the mismatch.
        # For this specific error, reassigning 'headers' directly is the targeted fix.
        pass # Original headers are already defined and unique.
    corrected_listOfFrames.append(df)

combinedDF = pd.concat(corrected_listOfFrames, axis=0, ignore_index=True)
combinedDF.shape

(65437, 6)

In [ ]:
combinedDF['Work Status'] = combinedDF['Work Status'].apply(str)

In [ ]:
combinedDF = combinedDF.drop_duplicates()
combinedDF = combinedDF.drop_duplicates(subset=['Practice', 'Claim ID', 'Worked Date','Worked By'])
combinedDF.shape

(65171, 6)

In [ ]:
# Create a regular expression pattern that matches any of the unwanted statuses
pattern = r"(Duplicate|Already done|Not Billed|None|Already worked|On Hold|Claim missing in CS|MSU Claim billed)"

In [ ]:
combinedDF = combinedDF.loc[~combinedDF['Work Status'].str.contains(pattern, flags=re.IGNORECASE, regex=True)]
combinedDF.shape

/tmp/ipython-input-833493247.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  combinedDF = combinedDF.loc[~combinedDF['Work Status'].str.contains(pattern, flags=re.IGNORECASE, regex=True)]


(48697, 6)

In [ ]:
combinedDF = combinedDF.loc[~combinedDF['Work Status'].isnull()]
combinedDF = combinedDF[combinedDF['Work Status'].notnull() & (combinedDF['Work Status'] != '')]
combinedDF.shape

(46806, 6)

In [ ]:
combinedDF = combinedDF.reset_index()

In [ ]:
combinedDF = combinedDF.loc[~combinedDF['Worked By'].isnull()]
combinedDF = combinedDF[combinedDF['Worked By'].notnull() & (combinedDF['Worked By'] != '')]
combinedDF.shape

(46806, 7)

In [ ]:
combinedDF = combinedDF.sort_values('Worked Date')
combinedDF = combinedDF.fillna("")
combinedDF.shape

(46806, 7)

In [ ]:
combinedDF['Worked Date'] = pd.to_datetime(combinedDF['Worked Date'],errors = 'coerce')
combinedDF.shape

(46806, 7)

In [ ]:
combinedDF = combinedDF.loc[~combinedDF['Worked Date'].isnull()]
combinedDF = combinedDF[combinedDF['Worked Date'].notnull() & (combinedDF['Worked Date'] != '')]
combinedDF.shape

(46793, 7)

In [ ]:
combinedDF = combinedDF.loc[combinedDF['Worked Date']>= '2025-01-01']
combinedDF.shape

(46793, 7)

In [ ]:
combinedDF['Worked Date'] = combinedDF['Worked Date'].dt.date

In [ ]:
combinedDF = combinedDF[['Work Type', 'Practice', 'Claim ID', 'Worked By', 'Work Status', 'Worked Date']]

In [ ]:
combinedDF['Claim ID'] = pd.to_numeric(combinedDF['Claim ID'], errors='coerce').astype('Int64')

In [ ]:
combinedDF = combinedDF.sort_values('Worked Date')

In [ ]:
combinedDF

,Work Type,Practice,Claim ID,Worked By,Work Status,Worked Date
0,After Transmit,1st Family Dental,207717,Anandu M,Billed,2025-11-03
752,After Transmit,United Dental Corporation,89783,Afzana Hussain,Billed,2025-11-03
751,After Transmit,United Dental Corporation,89782,Afzana Hussain,Billed,2025-11-03
750,After Transmit,United Dental Corporation,89736,Reshma B L,Billed,2025-11-03
749,After Transmit,United Dental Corporation,89735,Afzana Hussain,Billed,2025-11-03
...,...,...,...,...,...,...
46380,After Transmit,River Vista Dentistry,7415,Sayed Ahammed,Billed,2026-01-06
46381,After Transmit,River Vista Dentistry,7416,Sayed Ahammed,Billed,2026-01-06
46382,After Transmit,River Vista Dentistry,7417,Sayed Ahammed,Billed,2026-01-06
46373,After Transmit,River Vista Dentistry,7408,Sayed Ahammed,Billed,2026-01-06


In [ ]:
combinedDF['Worked Date'] = combinedDF['Worked Date'].apply(str)
##combinedDF['Claim ID'] = combinedDF['Claim ID'].apply(str)

In [ ]:
status_list = ['Manual Exception', 'Tickets', 'After Transmit']

In [ ]:
combinedDF

,Work Type,Practice,Claim ID,Worked By,Work Status,Worked Date
0,After Transmit,1st Family Dental,207717,Anandu M,Billed,2025-11-03
752,After Transmit,United Dental Corporation,89783,Afzana Hussain,Billed,2025-11-03
751,After Transmit,United Dental Corporation,89782,Afzana Hussain,Billed,2025-11-03
750,After Transmit,United Dental Corporation,89736,Reshma B L,Billed,2025-11-03
749,After Transmit,United Dental Corporation,89735,Afzana Hussain,Billed,2025-11-03
...,...,...,...,...,...,...
46380,After Transmit,River Vista Dentistry,7415,Sayed Ahammed,Billed,2026-01-06
46381,After Transmit,River Vista Dentistry,7416,Sayed Ahammed,Billed,2026-01-06
46382,After Transmit,River Vista Dentistry,7417,Sayed Ahammed,Billed,2026-01-06
46373,After Transmit,River Vista Dentistry,7408,Sayed Ahammed,Billed,2026-01-06


In [ ]:
# Remove rows where 'Work Type' is NaN or empty before pivoting
combinedDF = combinedDF[combinedDF['Work Type'].notnull() & (combinedDF['Work Type'] != '')]
combinedDF = combinedDF[combinedDF['Work Type'] != 'Pre-Auth']



# Pivot the DataFrame to get Work Type as columns, aggregated by Worked Date and Worked By
pivot_df = combinedDF.pivot_table(
    index=['Worked Date', 'Worked By'],    # Set both 'Worked Date' and 'Worked By' as indices
    columns='Work Type',                   # Set 'Work Type' as columns
    aggfunc='size',                        # Count the occurrences of each Work Type
    fill_value=0                           # Fill missing values with 0
)

# Remove any blank column caused by multi-indexing in the 'columns' level
pivot_df.columns.name = None  # This removes the 'Work Type' name in the column headers

# Reset index if you prefer a flat DataFrame
pivot_df = pivot_df.reset_index()

# View the reshaped DataFrame
#print(pivot_df)

In [ ]:
for status in status_list:
    combinedDF[status] = combinedDF['Work Status'].apply(lambda x: 1 if x == status else 0)

In [ ]:
combinedDFPivot = pivot_df.fillna(0)

In [ ]:
combinedDFPivot["Worked By"] = combinedDFPivot["Worked By"].str.strip()

In [ ]:
combinedDFPivot

,Worked Date,Worked By,After Transmit,Manual Exception,Tickets
0,2025-11-03,Afzana Hussain,63,0,0
1,2025-11-03,Akhil Gopal,57,0,0
2,2025-11-03,Anandu M,63,0,0
3,2025-11-03,Anuja K A R,61,0,0
4,2025-11-03,Braison L H,80,13,0
...,...,...,...,...,...
603,2026-01-06,Sayed Ahammed,59,0,0
604,2026-01-06,Shiva G,0,0,4
605,2026-01-06,Sreehari,78,1,0
606,2026-01-06,Sreejith S,0,64,0


In [ ]:
combinedDFPivot = [combinedDFPivot.columns.to_list()] + combinedDFPivot.to_numpy().tolist()

In [ ]:
wsMaster = gc.open_by_key("1ffYld-zvM4kMWB3c47GVqsmgoU8S2jPrwUG4U6fWXY8").worksheet("Claims Data")
wsMaster.clear()
wsMaster.update("A1",combinedDFPivot,value_input_option="USER_ENTERED")

/tmp/ipython-input-4217801965.py:3: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  wsMaster.update("A1",combinedDFPivot,value_input_option="USER_ENTERED")


{'spreadsheetId': '1ffYld-zvM4kMWB3c47GVqsmgoU8S2jPrwUG4U6fWXY8',
 'updatedRange': "'Claims Data'!A1:E609",
 'updatedRows': 609,
 'updatedColumns': 5,
 'updatedCells': 3045}